# GA4 Analytics Agent

The GA4 Analytics Agent is an AI-powered analytics assistant that queries Google Analytics 4, Google Search Console, and real-time visitor data through natural language. Ask for a traffic overview, top pages, search queries, or live visitor counts — and the agent pulls the numbers directly from your property.

In [1]:
!pip install -q aixplain google-analytics-data google-api-python-client google-auth

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
embedchain 0.1.123 requires rich<14.0.0,>=13.7.0, but you have rich 14.2.0 which is incompatible.


## Add your API keys

Get your aiXplain access key from the [Integrations](https://platform.aixplain.com/account/integrations) page.

For Google credentials:
1. Create a [Google Cloud project](https://console.cloud.google.com/) and enable the **Analytics Data API**, **Search Console API**, and **Indexing API**.
2. Create a **service account**, download the JSON key, and copy the `client_email` and `private_key` values.
3. In GA4 → Admin → Property Access Management, add the service account email as a **Viewer**.
4. In Google Search Console → Settings → Users and permissions, add the service account email.
5. Your GA4 Property ID is found in GA4 → Admin → Property Settings (numeric ID, e.g. `123456789`).

In [ ]:
AixplainAPIKey       = "<YOUR_AIXPLAIN_API_KEY>"
Ga4PropertyId        = "<YOUR_GA4_PROPERTY_ID>"        # numeric, e.g. "123456789"
Ga4ClientEmail       = "<YOUR_SERVICE_ACCOUNT_EMAIL>"  # e.g. "myapp@project.iam.gserviceaccount.com"
Ga4PrivateKey        = "<YOUR_SERVICE_ACCOUNT_PRIVATE_KEY>"  # full PEM key, including \n characters
SearchConsoleSiteUrl = "<YOUR_SEARCH_CONSOLE_SITE_URL>"  # e.g. "https://example.com/"

import os
os.environ["TEAM_API_KEY"]              = AixplainAPIKey
os.environ["GA4_PROPERTY_ID"]           = Ga4PropertyId
os.environ["GA4_CLIENT_EMAIL"]          = Ga4ClientEmail
os.environ["GA4_PRIVATE_KEY"]           = Ga4PrivateKey
os.environ["SEARCH_CONSOLE_SITE_URL"]   = SearchConsoleSiteUrl

In [3]:
import inspect
from aixplain import Aixplain

aix = Aixplain(AixplainAPIKey)

# What is the GA4 Analytics Agent?

The goal of this agent is to make web analytics conversational. Instead of navigating dashboards, you ask in plain English and the agent queries your GA4 property and Search Console directly.

## Agent Architecture

The agent uses three custom script tools:

- **GA4 Report** — queries the Analytics Data API for traffic overviews, content performance, traffic source breakdowns, and user behaviour metrics across any date range.
- **GA4 Realtime** — fetches live active users and the pages they are currently viewing.
- **Search Console Report** — retrieves top search queries and page performance (impressions, clicks, CTR, average position) from Google Search Console.

## Agent Workflow

1. **Understand the request** — determine whether it needs historical GA4 data, real-time data, or Search Console data.
2. **Call the right tool** — route to GA4 Report, Realtime, or Search Console accordingly.
3. **Summarise** — present the numbers clearly, highlight anomalies, and suggest follow-up questions.

In [4]:
def get_ga4_report(report_type: str = "overview", date_range: str = "30d") -> str:
    """
    Queries Google Analytics 4 for website traffic and performance data.

    Parameters:
        report_type (str): Type of report to run:
            - 'overview'  — sessions, users, page views, bounce rate, avg session duration
            - 'content'   — top pages by views with engagement rate
            - 'traffic'   — sessions by traffic source and medium
            - 'behavior'  — engagement metrics by device category and country
        date_range (str): Date range as shorthand ('7d', '30d', '90d') or
                          ISO range 'YYYY-MM-DD:YYYY-MM-DD'. Default: '30d'.

    Returns:
        str: Formatted analytics report.
    """
    import os
    import json
    from google.analytics.data_v1beta import BetaAnalyticsDataClient
    from google.analytics.data_v1beta.types import (
        DateRange, Dimension, Metric, RunReportRequest, OrderBy
    )
    from google.oauth2 import service_account

    property_id  = os.environ.get("GA4_PROPERTY_ID")
    client_email = os.environ.get("GA4_CLIENT_EMAIL")
    private_key  = os.environ.get("GA4_PRIVATE_KEY")

    if not all([property_id, client_email, private_key]):
        return "Error: GA4_PROPERTY_ID, GA4_CLIENT_EMAIL, and GA4_PRIVATE_KEY must all be set."

    # Parse date range
    shorthand = {"7d": "7daysAgo", "30d": "30daysAgo", "90d": "90daysAgo"}
    if ":" in date_range:
        start_date, end_date = date_range.split(":", 1)
    elif date_range in shorthand:
        start_date, end_date = shorthand[date_range], "today"
    else:
        start_date, end_date = "30daysAgo", "today"

    # Service account credentials
    creds_info = {
        "type": "service_account",
        "client_email": client_email,
        "private_key": private_key.replace("\\n", "\n"),
        "token_uri": "https://oauth2.googleapis.com/token",
    }
    credentials = service_account.Credentials.from_service_account_info(
        creds_info,
        scopes=["https://www.googleapis.com/auth/analytics.readonly"],
    )
    client = BetaAnalyticsDataClient(credentials=credentials)

    # Report configurations
    configs = {
        "overview": {
            "dimensions": [],
            "metrics": ["sessions", "totalUsers", "screenPageViews", "bounceRate",
                        "averageSessionDuration", "newUsers"],
            "order_by": None,
            "limit": 1,
        },
        "content": {
            "dimensions": ["pagePath", "pageTitle"],
            "metrics": ["screenPageViews", "engagementRate", "averageSessionDuration"],
            "order_by": "screenPageViews",
            "limit": 20,
        },
        "traffic": {
            "dimensions": ["sessionSource", "sessionMedium"],
            "metrics": ["sessions", "totalUsers", "bounceRate"],
            "order_by": "sessions",
            "limit": 20,
        },
        "behavior": {
            "dimensions": ["deviceCategory", "country"],
            "metrics": ["sessions", "engagementRate", "averageSessionDuration"],
            "order_by": "sessions",
            "limit": 20,
        },
    }

    cfg = configs.get(report_type, configs["overview"])

    try:
        request = RunReportRequest(
            property=f"properties/{property_id}",
            date_ranges=[DateRange(start_date=start_date, end_date=end_date)],
            dimensions=[Dimension(name=d) for d in cfg["dimensions"]],
            metrics=[Metric(name=m) for m in cfg["metrics"]],
            order_bys=(
                [OrderBy(metric=OrderBy.MetricOrderBy(metric_name=cfg["order_by"]), desc=True)]
                if cfg["order_by"] else []
            ),
            limit=cfg["limit"],
        )
        response = client.run_report(request)
    except Exception as e:
        return f"Error querying GA4: {e}"

    if not response.rows:
        return f"No data found for '{report_type}' in {date_range}."

    dim_headers = [h.name for h in response.dimension_headers]
    met_headers = [h.name for h in response.metric_headers]
    lines = [f"GA4 {report_type.upper()} REPORT ({date_range})\n{'─'*50}"]

    for row in response.rows:
        dims = {dim_headers[i]: row.dimension_values[i].value for i in range(len(dim_headers))}
        mets = {met_headers[i]: row.metric_values[i].value for i in range(len(met_headers))}
        if dims:
            lines.append("  ".join(f"{k}: {v}" for k, v in dims.items()))
        for k, v in mets.items():
            try:
                v_fmt = f"{float(v):,.2f}" if "." in v else f"{int(v):,}"
            except ValueError:
                v_fmt = v
            lines.append(f"  {k}: {v_fmt}")
        lines.append("")

    return "\n".join(lines)

In [5]:
def get_ga4_realtime() -> str:
    """
    Returns a real-time snapshot of active users currently on the site.

    Returns:
        str: Number of active users in the last 30 minutes, broken down by page and country.
    """
    import os
    from google.analytics.data_v1beta import BetaAnalyticsDataClient
    from google.analytics.data_v1beta.types import (
        Dimension, Metric, RunRealtimeReportRequest
    )
    from google.oauth2 import service_account

    property_id  = os.environ.get("GA4_PROPERTY_ID")
    client_email = os.environ.get("GA4_CLIENT_EMAIL")
    private_key  = os.environ.get("GA4_PRIVATE_KEY")

    if not all([property_id, client_email, private_key]):
        return "Error: GA4_PROPERTY_ID, GA4_CLIENT_EMAIL, and GA4_PRIVATE_KEY must all be set."

    creds_info = {
        "type": "service_account",
        "client_email": client_email,
        "private_key": private_key.replace("\\n", "\n"),
        "token_uri": "https://oauth2.googleapis.com/token",
    }
    credentials = service_account.Credentials.from_service_account_info(
        creds_info,
        scopes=["https://www.googleapis.com/auth/analytics.readonly"],
    )
    client = BetaAnalyticsDataClient(credentials=credentials)

    try:
        request = RunRealtimeReportRequest(
            property=f"properties/{property_id}",
            dimensions=[Dimension(name="unifiedPagePathScreen"), Dimension(name="country")],
            metrics=[Metric(name="activeUsers")],
            limit=20,
        )
        response = client.run_realtime_report(request)
    except Exception as e:
        return f"Error querying GA4 Realtime: {e}"

    if not response.rows:
        return "No active users right now."

    total = sum(int(row.metric_values[0].value) for row in response.rows)
    lines = [f"LIVE SNAPSHOT — {total} active user(s) in the last 30 minutes\n{'─'*50}"]
    for row in response.rows:
        page    = row.dimension_values[0].value
        country = row.dimension_values[1].value
        users   = row.metric_values[0].value
        lines.append(f"  {users} user(s)  |  {page}  |  {country}")

    return "\n".join(lines)

In [6]:
def get_search_console_report(report_type: str = "queries", date_range: str = "30d") -> str:
    """
    Queries Google Search Console for search performance data.

    Parameters:
        report_type (str): Type of report:
            - 'queries' — top search queries with clicks, impressions, CTR, and position
            - 'pages'   — top pages with clicks, impressions, CTR, and position
        date_range (str): Date range as shorthand ('7d', '30d', '90d') or
                          ISO range 'YYYY-MM-DD:YYYY-MM-DD'. Default: '30d'.

    Returns:
        str: Formatted Search Console report.
    """
    import os
    from datetime import date, timedelta
    from googleapiclient.discovery import build
    from google.oauth2 import service_account

    site_url     = os.environ.get("SEARCH_CONSOLE_SITE_URL")
    client_email = os.environ.get("GA4_CLIENT_EMAIL")
    private_key  = os.environ.get("GA4_PRIVATE_KEY")

    if not all([site_url, client_email, private_key]):
        return "Error: SEARCH_CONSOLE_SITE_URL, GA4_CLIENT_EMAIL, and GA4_PRIVATE_KEY must all be set."

    # Parse date range
    today = date.today()
    shorthand_days = {"7d": 7, "30d": 30, "90d": 90}
    if ":" in date_range:
        start_date, end_date = date_range.split(":", 1)
    elif date_range in shorthand_days:
        days = shorthand_days[date_range]
        start_date = (today - timedelta(days=days)).isoformat()
        end_date   = today.isoformat()
    else:
        start_date = (today - timedelta(days=30)).isoformat()
        end_date   = today.isoformat()

    creds_info = {
        "type": "service_account",
        "client_email": client_email,
        "private_key": private_key.replace("\\n", "\n"),
        "token_uri": "https://oauth2.googleapis.com/token",
    }
    credentials = service_account.Credentials.from_service_account_info(
        creds_info,
        scopes=["https://www.googleapis.com/auth/webmasters.readonly"],
    )

    dimension = "query" if report_type == "queries" else "page"

    try:
        service = build("searchconsole", "v1", credentials=credentials)
        response = service.searchanalytics().query(
            siteUrl=site_url,
            body={
                "startDate": start_date,
                "endDate": end_date,
                "dimensions": [dimension],
                "rowLimit": 25,
                "orderBy": [{"fieldName": "clicks", "sortOrder": "DESCENDING"}],
            },
        ).execute()
    except Exception as e:
        return f"Error querying Search Console: {e}"

    rows = response.get("rows", [])
    if not rows:
        return f"No Search Console data found for '{report_type}' in {date_range}."

    label = "Query" if report_type == "queries" else "Page"
    lines = [f"SEARCH CONSOLE {report_type.upper()} ({date_range})\n{'─'*50}"]
    for i, row in enumerate(rows, start=1):
        key     = row["keys"][0]
        clicks  = int(row.get("clicks", 0))
        imps    = int(row.get("impressions", 0))
        ctr     = row.get("ctr", 0) * 100
        pos     = row.get("position", 0)
        lines.append(
            f"[{i:02d}] {label}: {key}\n"
            f"      Clicks: {clicks:,}  |  Impressions: {imps:,}  "
            f"|  CTR: {ctr:.1f}%  |  Avg Position: {pos:.1f}"
        )

    return "\n".join(lines)

In [7]:
SCRIPT_INTEGRATION_ID = "688779d8bfb8e46c273982ca"

# Inject all credentials into each tool's source — os.environ is not
# available in aiXplain's cloud tool runtime, so the keys must be embedded.
def _inject(fn, replacements):
    src = inspect.getsource(fn)
    for placeholder, value in replacements.items():
        src = src.replace(placeholder, repr(value))
    return src

cred_map = {
    'os.environ.get("GA4_PROPERTY_ID")':         Ga4PropertyId,
    'os.environ.get("GA4_CLIENT_EMAIL")':         Ga4ClientEmail,
    'os.environ.get("GA4_PRIVATE_KEY")':          Ga4PrivateKey,
    'os.environ.get("SEARCH_CONSOLE_SITE_URL")':  SearchConsoleSiteUrl,
}

ga4_report_tool = aix.Tool(
    name="GA4 Report",
    integration=SCRIPT_INTEGRATION_ID,
    config={"code": _inject(get_ga4_report, cred_map), "function_name": "get_ga4_report"},
)
ga4_report_tool.save()
print(f"Created: {ga4_report_tool.name}")

ga4_realtime_tool = aix.Tool(
    name="GA4 Realtime",
    integration=SCRIPT_INTEGRATION_ID,
    config={"code": _inject(get_ga4_realtime, cred_map), "function_name": "get_ga4_realtime"},
)
ga4_realtime_tool.save()
print(f"Created: {ga4_realtime_tool.name}")

search_console_tool = aix.Tool(
    name="Search Console Report",
    integration=SCRIPT_INTEGRATION_ID,
    config={"code": _inject(get_search_console_report, cred_map), "function_name": "get_search_console_report"},
)
search_console_tool.save()
print(f"Created: {search_console_tool.name}")

Created: GA4 Report
Created: GA4 Realtime
Created: Search Console Report


# Build Your Agent

To create an agent, define:

* A unique **name** and **description** for its purpose.
* The **tools** it will use — here, GA4 Report, GA4 Realtime, and Search Console Report.
* The **instructions** that guide the agent's analysis and reporting behaviour.

In [8]:
ga4_agent = aix.Agent(
    name="GA4 Analytics Agent",
    description="Analytics assistant that queries Google Analytics 4 and Search Console to deliver traffic overviews, content performance, real-time visitors, and SEO insights.",
    instructions="""
    You are an expert web analytics assistant with access to Google Analytics 4 and Google Search Console.
    You have three tools — choose the most appropriate one for each request:

    - GA4 Report: for historical traffic data. Use report_type:
        'overview'  — total sessions, users, page views, bounce rate, session duration
        'content'   — top pages by views
        'traffic'   — sessions by source and medium
        'behavior'  — engagement by device and country
      Date range accepts '7d', '30d', '90d', or 'YYYY-MM-DD:YYYY-MM-DD'.

    - GA4 Realtime: for the current live visitor count and what pages they are on.
      Use this when the user asks 'right now', 'live', or 'currently'.

    - Search Console Report: for SEO data. Use report_type:
        'queries' — top search queries with clicks, impressions, CTR, position
        'pages'   — top pages in search results

    Guidelines:
    - Always include the date range in your answer so the user knows the timeframe.
    - Highlight the most significant numbers — biggest traffic sources, top page, best-ranking query.
    - For comparisons (e.g. 'vs last month'), call GA4 Report twice with different date ranges.
    - If numbers look unusual, note it and suggest what might explain it.
    - Keep responses concise and actionable — surface insights, not just raw tables.
    """,
    tools=[ga4_report_tool, ga4_realtime_tool, search_console_tool],
)
ga4_agent.save()
print(f"Agent created: {ga4_agent.id}")

Agent created: 69c541609e1dff6231a1b77c


# Run your Agent

To ensure your agent is performing as expected, run it using the `run` method with sample inputs. Analyze the outputs, verify their accuracy, and debug any issues by inspecting the agent's configurations, tools, and instructions.

In [9]:
# Traffic overview for the last 30 days
result = ga4_agent.run("Give me a traffic overview for the last 30 days")
print(result.data.output)

I am unable to provide the requested traffic overview for the last 30 days due to technical issues with the tools. I recommend checking Google Analytics and Search Console directly for the most accurate and up-to-date information.


In [ ]:
# Top pages
result2 = ga4_agent.run("What are my top 10 pages by views this week?")
print(result2.data.output)

In [11]:
# Real-time visitors
result3 = ga4_agent.run("How many people are on my site right now?")
print(result3.data.output)

I am currently unable to retrieve the number of visitors on your site due to technical issues with the analytics tools. Please try again later.


In [ ]:
# Search Console — top queries
result4 = ga4_agent.run("What search queries are driving the most clicks from Google?")
print(result4.data.output)

In [ ]:
# Multi-turn: compare periods
result5 = ga4_agent.run(
    "How does traffic compare to the previous 30-day period?",
    session_id=result.data.session_id
)
print(result5.data.output)

# Save the Agent

Once you are happy with your agent, save it to access the agent endpoints.

In [ ]:
ga4_agent.save()

aiXplain empowers you to seamlessly build, customize, and deploy intelligent agents tailored to your specific needs. Whether you're creating standalone agents or advanced multi-agent systems, the platform provides a robust toolkit for integrating cutting-edge AI capabilities into your workflows. To learn more, visit [aiXplain](https://aixplain.com).